In [1]:
# ════════════════════════════════════════════════════════════════════════════
# RAG Pipeline Configuration & Imports
# ════════════════════════════════════════════════════════════════════════════

# Standard Library Imports
import json
import logging
import math
import os
import pickle
from dataclasses import dataclass
from datetime import datetime
from enum import Enum
from functools import partial
from typing import Any, Dict, List, Optional
from uuid import uuid4

# Third-party Imports
from pydantic import BaseModel, ConfigDict, Field
import openai

# Project Imports
from config import CONFIG, TextNode, setup, SearchResult
from prompts import RAG_EVALUATOR_SYSTEM
from all_functionality import embed_text, async_chat_wrapper

setup()

logger = logging.getLogger(__name__)


Evaluation functions defined.
Visualization function defined.


#### Corpora txt

In [2]:
with open('corpora.txt', 'r', encoding='utf-8') as f:
    corpora = f.read().splitlines()
    corpora = [corpus.strip() for corpus in corpora if corpus.strip()]
corpora_text = "\n".join(corpora) if isinstance(corpora, list) else str(corpora)

In [3]:
list(corpora_text.split("```"))[:2]

['## Tasks data source description\n{\'object\': \'data_source\',\n\'id\': \'0c0c2dea-6c50-4abb-b720-52f00d875899\',\n\'cover\': None,\n\'icon\': {\'type\': \'emoji\', \'emoji\': \'📰\'},\n\'created_time\': \'2024-08-11T10:28:00.000Z\',\n\'created_by\': {\'object\': \'user\',\n\'id\': \'141465d0-0b53-4ca9-bfa1-404af1627be8\'},\n\'last_edited_by\': {\'object\': \'user\',\n\'id\': \'141465d0-0b53-4ca9-bfa1-404af1627be8\'},\n\'last_edited_time\': \'2026-02-21T09:10:00.000Z\',\n\'title\': [{\'type\': \'text\',\n\'text\': {\'content\': \'Tasks\', \'link\': None},\n\'annotations\': {\'bold\': False,\n\'italic\': False,\n\'strikethrough\': False,\n\'underline\': False,\n\'code\': False,\n\'color\': \'default\'},\n\'plain_text\': \'Tasks\',\n\'href\': None}],\n\'description\': [],\n\'is_inline\': False,\n\'properties\': {\'Project/status\': {\'id\': \'%3Dq%5CG\',\n\'name\': \'Project/status\',\n\'description\': None,\n\'type\': \'rollup\',\n\'rollup\': {\'rollup_property_name\': \'Status\',\n\'

In [4]:
# length
len(corpora_text)

1124615

### RAG

#### Hierarchical Text Processing

**Architecture:**
- **TextNode Pydantic Model**: Text chunks with unique IDs and embeddings
- **Recursive Splitting**: Multi-layer hierarchy based on token targets per layer
- **Storage Dictionary**: Maps `id → {parent_id, text, layer}` for rapid reconstruction
- **Leaf-Only Vectorization**: Embeddings computed **only** for deepest nodes (cost optimization)

**How it works:**
1. Recursively splits text into `n_layers` hierarchical levels
2. Each layer has a target token count (`tokens_per_layer`)
3. Storage dict maintains parent-child relationships (lazy reconstruction)
4. Leaf nodes (last layer) are embedded for semantic search
5. Returns both storage dict and vectorized TextNode list

**Usage:**
```python
storage, corpora_vectorized = process_corpora_to_hierarchy(
    text=larger_text,
    n_layers=3,
    tokens_per_layer=[8000, 5000, 500]
)
# storage: {id: {parent_id, text, layer, embedding_id}}
# corpora_vectorized: [TextNode(id, embedding), ...]
```

**Cost Benefits:**
- Embeddings only for leaves (~500 tokens each) vs all nodes
- Parent nodes remain unembedded for hierarchical search
- Token efficiency: structured splitting prevents over-fragmentation

In [5]:
#### Hierarchical Data Processor


def estimate_tokens(text: str) -> int:
    """
    Rough token estimation. Assumes ~4 chars per token (typical for English).
    For precise counts, would use tiktoken or similar.
    """
    chars_per_token = 4
    return max(1, len(text) // chars_per_token)


def split_text_recursive(
    text: str,
    n_layers: int,
    tokens_per_layer: List[int],
    parent_id: Optional[str] = None,
    storage: Optional[Dict[str, Dict[str, Any]]] = None,
    current_layer: int = 0,
) -> Dict[str, Dict[str, Any]]:
    """
    Recursively split text into a hierarchical structure.
    
    Layer 0 is the first split. Corpora is NOT stored; only hierarchical chunks are stored.
    
    Args:
        text: Text to split at this layer
        n_layers: Total number of layers (0 to n_layers-1)
        tokens_per_layer: Target token counts for EACH layer [layer_0_target, layer_1_target, ...]
        parent_id: Parent node ID (None for root corpora, not stored)
        storage: Dictionary to accumulate all nodes {id: {parent_id, text, layer}}
        current_layer: Current layer (0 = first split)
        
    Returns:
        Storage dictionary with all layer nodes
    """
    if storage is None:
        storage = {}
    
    # Base case: reached max layers
    if current_layer >= n_layers:
        return storage
    
    # Get target tokens for THIS layer
    target_tokens = tokens_per_layer[current_layer]
    text_length = estimate_tokens(text)
    
    # Case 1: Text fits in target for this layer
    if text_length <= target_tokens:
        # Create single node for this layer (don't store if it's root with parent_id=None)
        if parent_id is not None or current_layer > 0:
            node_id = str(uuid4())
            storage[node_id] = {
                "parent_id": parent_id,
                "text": text,
                "layer": current_layer,
            }
        return storage
    
    # Case 2: Text needs to split at this layer
    num_chunks = max(1, (text_length + target_tokens - 1) // target_tokens)
    chunk_size = len(text) // num_chunks
    chunks = []
    current_pos = 0
    
    for i in range(num_chunks):
        if i == num_chunks - 1:
            # Last chunk gets remainder
            chunk = text[current_pos:]
        else:
            # Find break point at space
            chunk_end = current_pos + chunk_size
            if chunk_end < len(text):
                last_space = text.rfind(' ', current_pos, chunk_end)
                if last_space > current_pos:
                    chunk_end = last_space
            chunk = text[current_pos:chunk_end]
            current_pos = chunk_end + 1
        
        if chunk.strip():
            chunks.append(chunk.strip())
    
    # Create nodes for each chunk and recurse to next layer
    for chunk in chunks:
        node_id = str(uuid4())
        storage[node_id] = {
            "parent_id": parent_id,
            "text": chunk,
            "layer": current_layer,
        }
        
        # Recurse to next layer if not at deepest
        if current_layer < n_layers - 1:
            split_text_recursive(
                chunk,
                n_layers=n_layers,
                tokens_per_layer=tokens_per_layer,
                parent_id=node_id,
                storage=storage,
                current_layer=current_layer + 1,
            )
    
    return storage


def process_corpora_to_hierarchy(
    text: str,
    n_layers: int,
    tokens_per_layer: List[int],
) -> tuple[Dict[str, Dict[str, Any]], List[TextNode]]:
    """
    Main entry point: Process text into hierarchical structure and embed leaf nodes.
    
    Args:
        text: Raw corpora text (NOT stored as a node)
        n_layers: Number of hierarchy layers
        tokens_per_layer: Target token counts per layer (must match n_layers length)
        
    Returns:
        Tuple of:
        - storage: Dict[id -> {parent_id, text, layer}] for reconstruction
        - corpora_vectorized: List[TextNode] with embeddings only for leaf nodes
    """
    # Validate inputs
    if len(tokens_per_layer) != n_layers:
        raise ValueError(f"tokens_per_layer must have {n_layers} elements, got {len(tokens_per_layer)}")
    
    # Build hierarchical structure (corpora not stored as node)
    storage = split_text_recursive(
        text=text,
        n_layers=n_layers,
        tokens_per_layer=tokens_per_layer,
    )
    
    # Identify leaf nodes (nodes with no children)
    all_ids = set(storage.keys())
    parent_ids = {node["parent_id"] for node in storage.values() if node["parent_id"]}
    leaf_ids = all_ids - parent_ids
    
    # Embed only leaf nodes
    corpora_vectorized: List[TextNode] = []
    
    for leaf_id in leaf_ids:
        leaf_text = storage[leaf_id]["text"]
        
        # Get embedding for this leaf node using the embedding wrapper
        embedding = embed_text(leaf_text)
        
        # Create TextNode with embedding
        node = TextNode(
            id=leaf_id,
            embedding=embedding,
        )
        corpora_vectorized.append(node)
        
        # Update storage to include embedding reference
        storage[leaf_id]["embedding_id"] = leaf_id
    
    logger.info(
        "Hierarchy created: %d total nodes, %d leaf nodes (embedded)",
        len(storage),
        len(corpora_vectorized),
    )
    
    return storage, corpora_vectorized


#### RAG RUN

In [7]:
#### RAG RUN

# Create vectors directory
os.makedirs("vectors", exist_ok=True)

storage_path = "vectors/storage.pkl"
vectors_path = "vectors/corpora_vectorized.pkl"

if os.path.exists(storage_path) and os.path.exists(vectors_path):
    raise FileExistsError(
        f"Storage and vectors already exist at {storage_path} and {vectors_path}. ")

### Hierarchical Text Processing Demo

# Example: Process corpora into hierarchical structure
# Token targets for each layer: Layer 1: ~5000 tokens, Layer 2 (leaves): ~500 tokens

tokens_per_layer = [5000, 500]
n_layers = len(tokens_per_layer)

# Process (uncomment to run and embed; creates many API calls)
# Expects 'corpora_text' to be loaded/defined before execution
if 'corpora_text' in globals():
    storage, corpora_vectorized = process_corpora_to_hierarchy(
        text=corpora_text,
        n_layers=n_layers,
        tokens_per_layer=tokens_per_layer,
    )
    print(f"Storage nodes: {len(storage)}, Vectorized leaves: {len(corpora_vectorized)}")

    with open(storage_path, "wb") as f:
        pickle.dump(storage, f, protocol=pickle.HIGHEST_PROTOCOL)

    with open(vectors_path, "wb") as f:
        pickle.dump(corpora_vectorized, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"Saved: {storage_path} and {vectors_path}")
else:
    print("Warning: 'corpora_text' not found in globals. Please load corpora before running.")


2026-03-02 12:41:14 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/embeddings "HTTP/1.1 200 OK"
2026-03-02 12:41:15 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/embeddings "HTTP/1.1 200 OK"
2026-03-02 12:41:15 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/embeddings "HTTP/1.1 200 OK"
2026-03-02 12:41:16 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/embeddings "HTTP/1.1 200 OK"
2026-03-02 12:41:16 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/embeddings "HTTP/1.1 200 OK"
2026-03-02 12:41:17 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/embeddings "HTTP/1.1 200 OK"
2026-03-02 12:41:17 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/embeddings "HT

Storage nodes: 628, Vectorized leaves: 571
Saved: vectors/storage.pkl and vectors/corpora_vectorized.pkl


#### RAG Retrieval Evaluator

Judges **how well retrieved chunks answer a query** across four graders:

| Grader | Type | Description |
|---|---|---|
| `topic_matched` | `bool` | Retrieved content is on-topic for the query |
| `objects_coverage` | `float 0–1` | Fraction of key objects mentioned in the query that appear in results |
| `endpoint_presence` | `present / not_present / not_needed` | Whether the relevant API endpoint is surfaced |
| `properties_discussed` / `properties_total` | `int N / int K` | N properties covered from K that exist (can be 0/0) |

The judge outputs a structured JSON rating plus a 2-sentence critique.

In [ ]:
#### RAG Retrieval Evaluator


class RagEvaluator:
    """
    LLM-judge that grades RAG retrieval quality for a given query.

    Returns the raw dict from the LLM — keys depend on the prompt in use,
    so no Pydantic model is enforced here. Callers can extract/validate
    whatever fields their prompt defines.

    Args:
        model_size:   Model size for evaluation (None = use CONFIG default).
        temperature:  Sampling temperature (None = use default 0.1 for deterministic grading).
        max_chunks:   Cap on the number of retrieved chunks sent to the judge
                      (None = use default 8).
    """

    def __init__(
        self,
        model_size: Optional[str] = None,
        temperature: Optional[float] = None,
        max_chunks: Optional[int] = None,
    ) -> None:
        self.model_size = model_size if model_size is not None else CONFIG.default_model
        self.temperature = temperature if temperature is not None else 0.1
        self.max_chunks = max_chunks if max_chunks is not None else 8

    # ── public API ──────────────────────────────────────────────────────────

    async def evaluate(
        self,
        query:   str,
        results: List[SearchResult],
    ) -> Dict[str, Any]:
        """
        Grade a (query, retrieved-results) pair.

        Returns the raw LLM JSON dict — structure depends on the prompt used.
        """
        chunks = results[: self.max_chunks]
        context_block = self._format_chunks(chunks)

        messages = [
            {
                "role": "user",
                "content": (
                    f"{RAG_EVALUATOR_SYSTEM}\n\n"
                    f"## Query\n{query}\n\n"
                    f"## Retrieved Chunks\n{context_block}"
                ),
            },
        ]

        raw: Dict[str, Any] = await async_chat_wrapper(
            messages=messages,
            max_tokens=512,
            temperature=self.temperature,
            json_output=True,
            model_size=self.model_size,
        )

        return raw if isinstance(raw, dict) else {}

    async def evaluate_batch(
        self,
        pairs: List[Dict[str, Any]],
    ) -> List[Dict[str, Any]]:
        """
        Evaluate multiple (query, results) pairs.

        Returns a list of raw dicts in the same order as input pairs.
        """
        return [await self.evaluate(p["query"], p["results"]) for p in pairs]

    # ── helpers ─────────────────────────────────────────────────────────────

    @staticmethod
    def _format_chunks(chunks: List[SearchResult]) -> str:
        """Render retrieved chunks into a numbered, score-annotated block."""
        if not chunks:
            return "(no results retrieved)"
        lines = []
        for i, r in enumerate(chunks, 1):
            preview = r.text.replace("\n", " ").strip()
            lines.append(
                f"[{i}] score={r.score:.4f}  layer={r.layer}\n{preview}"
            )
        return "\n\n".join(lines)
